In [1]:
import pandas as pd
import re
from pprint import pprint
import numpy as np
cgg_path = r'c:\Users\glj523\Downloads\LAST_VERSION_OF_CGG3_20240410.xlsx'
cgg_df_orig = pd.read_excel(cgg_path, sheet_name='Sediment_Water', dtype=str)

Functions

In [2]:
def coord_format_checker(coord, axis):
    regex_dd = {'latitude': r"[-+]?\s*(\d{1,2}\.\d+)", 
                'longitude': r"[-+]?\s*(\d{1,3}\.\d+)"}

    regex_dms = {'latitude': r"(\d{1,2})\s+(\d{1,2})\s+(\d{1,2}(\.\d+)?)\s*([nsNS])",
                'longitude': r"(\d{1,3})\s+(\d{1,2})\s+(\d{1,2}(\.\d+)?)\s*([ewEW])"}

    regex_ddm = {'latitude': r"(\d{1,2})\s+(\d{1,2}(\.\d+)?)\s+([nsNS])", 
                'longitude': r"(\d{1,3})\s+(\d{1,2}(\.\d+)?)\s+([ewEW])"}

    if re.match(regex_dd[axis], coord):
        return 'DD'
    elif re.match(regex_dms[axis], coord):
        return 'DMS'
    elif re.match(regex_ddm[axis], coord):
        return 'DDM'
    else:
        return 'invalid format'
    
def coord_format_checker_cleaning(coord, axis):
    regex_dd = {'latitude': r"^[nNsS]?\s*\d{1,2}(?:\.\d+)?$", 
                'longitude': r"^[eEwW]?\s*\d{1,3}(?:\.\d+)?$"}

    regex_dms = {'latitude': r"^[nNsS]?\s*(\d{1,2})\s+(\d{1,2})\s+(\d{1,2})(?:\.\d+)?$",
                'longitude': r"^[eEwW]?\s*(\d{1,3})\s+(\d{1,2})\s+(\d{1,2})(?:\.\d+)?$"}

    regex_ddm = {'latitude': r"^[nNsS]?\s*(\d{1,2})\s+(\d{1,2})(?:\.\d+)?$", 
                'longitude': r"^[eEwW]?\s*(\d{1,3})\s+(\d{1,2})(?:\.\d+)?$"}

    if re.match(regex_dd[axis], coord):
        return 'DD'
    elif re.match(regex_dms[axis], coord):
        return 'DMS'
    elif re.match(regex_ddm[axis], coord):
        return 'DDM'
    else:
        return 'invalid format'
    
def move_letters_to_front(s):
    # Ensure we're working with a string (in case of NaN or other types)
    s = str(s)
    # Extract all letters (A-Z and a-z)
    letters = "".join(re.findall(r"[A-Za-z]", s))
    # Extract all non-letter characters
    non_letters = "".join(re.findall(r"[^A-Za-z]", s))
    # Concatenate letters first, then non-letters
    return letters + non_letters

def move_letters_to_back(s):
    # Ensure we're working with a string (in case of NaN or other types)
    s = str(s)
    # Extract all letters (A-Z and a-z)
    letters = "".join(re.findall(r"[A-Za-z]", s))
    # Extract all non-letter characters
    non_letters = "".join(re.findall(r"[^A-Za-z]", s))
    # Concatenate letters first, then non-letters
    return non_letters + letters

def dms_to_dd(degrees, minutes, seconds, direction):
    """
    Convert degrees, minutes, and seconds (DMS) to decimal degrees (DD).
    
    :param degrees: Degrees component
    :param minutes: Minutes component
    :param seconds: Seconds component
    :param direction: 'N', 'S', 'E', or 'W' for latitude/longitude
    :return: Decimal degree representation
    """
    dd = degrees + (minutes / 60) + (seconds / 3600)
    
    # If the direction is South or West, make it negative
    if direction in ['S', 'W']:
        dd *= -1
    
    return dd


Aggregate all values in the unamed columns to a single column where values from different columns are seperated by |

In [3]:
cgg_df = cgg_df_orig.copy()
cgg_df = cgg_df.map(lambda x: x.strip() if isinstance(x, str) else x)
cgg_df = cgg_df.replace(['', 'nan'], np.nan)
cgg_df = cgg_df.dropna(how='all', axis='columns')
unnamed_cols = cgg_df.columns[cgg_df.columns.str.startswith('Unnamed')]
cgg_df['from_unnamed_cols'] = cgg_df[unnamed_cols].agg(lambda x: ' | '.join(x.dropna().astype(str)), axis=1).str.strip()
cgg_df = cgg_df.replace('', np.nan)
cgg_df = cgg_df.drop(columns=unnamed_cols)

# Cleaning latitude

Standard cleaning


In [4]:
cgg_df['cleaned_lat'] = (cgg_df.Lat
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        )  

Identify unique formats

In [7]:
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats.unique().tolist()

['+12.12',
 nan,
 '+12',
 '12',
 '12.12',
 '12 12.12 N',
 '12°12′N',
 'N12',
 '12 12 12.12 N',
 "W12 12' 12.12''",
 '12.12 N',
 '12deg12\'12.12"S',
 "N 12º12.12'",
 'N12.12',
 '12.12N',
 "12°12'N",
 "12°12'12''N",
 '12.12°',
 "12°12'12''",
 '12°12\'12"',
 '-12.12',
 'N 12.12',
 '12.12°N',
 '12.12˚N',
 '12˚12\'12.12" N',
 "12 12' 12'' N",
 'N12° 12\' 12.12"',
 '12.12° N',
 '12° 12.12',
 '12°12\'12.12"N',
 '12°12\'12"N',
 '12o12.12',
 '12° 12\' 12.12" N',
 '12°12’12”',
 '12°12\'12.12"',
 "12° 12' 12.12'' N",
 "12° 12'12.12'' N",
 '12° 12.12´S',
 '12° 12’ 12.12” N',
 '12°12’12.12’’N',
 '12° 12.12 N',
 '12°12\'12.12"S',
 '12°12‘12‘‘ N',
 "12' 12.12''",
 '12° 12’ 12.12”',
 '12°12\'12" N',
 '12° 12´12´´N',
 '12°12.12’N',
 '12° 12′ 12″ N']

### Convert to standard characters and symbols
The bad characters were found from the above step

In [8]:
cgg_df['cleaned_lat'] = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

Put direction indicators (N, S, -, +) in a seperate column

In [9]:
cgg_df['lat_direction'] = (cgg_df.cleaned_lat
                           .map(lambda x: re.sub(r"[^a-zA-Z-+]", '', x), na_action='ignore')
                           )
#  Removes the direction from the cleaned_lat column, as now it is no longer needed
cgg_df.cleaned_lat = (cgg_df.cleaned_lat
                      .map(lambda x: re.sub(r"[a-zA-Z-+]", '', x), na_action='ignore')
                      .str.strip())

Identify unique formats again

In [10]:
from pprint import pprint
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats = pd.Series(lat_formats.dropna().unique())
print('With degree symbol')
pprint(lat_formats.dropna()[lat_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lat_formats.dropna()[~lat_formats.dropna().str.contains('°')].unique().tolist())

With degree symbol
["12°12'",
 '12°12\'12.12"',
 "12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12° 12\' 12.12"',
 '12° 12.12',
 '12°12.12',
 '12° 12\'12.12"',
 "12° 12.12'",
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12 12.12',
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"',
 '12\' 12.12"']


From the above it seems safe to assume that if
1. A coordinate contains 1 number with or without traling ° that the format is decimal degrees.
2. A coordinate contains 2 numbers seperated with with either a whitespace or ° or both that its formatted as degrees and minutes.
3. A coordinate contains 3 numbers, where first seperator is by ° or whitespace or both and second seperator is ' or whitespace or both, it is formatted as degree minute seconds.

We can formulate this using the following regular expressions:

In [11]:
dd_regex = r'''^\d{1,2}(\.\d+)?(°| °)?$'''
dm_regex = r'''^\d{1,2}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex = r'''^\d{1,2}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''

Checking the general formats 

In [12]:
def check_format(s: str):
    if re.match(dd_regex, s):
        return 'DD'
    elif re.match(dm_regex, s):
        return 'DM'
    elif re.match(dms_regex, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lat_formats.apply(check_format)

lat_with_formats = pd.DataFrame({'Lat': lat_formats, 'Format': classified_formats}).sort_values(by='Format')
lat_with_formats                           

,Lat,Format
0,12.12,DD
9,12.12°,DD
1,12,DD
2,12 12.12,DM
3,12°12',DM
15,12° 12.12',DM
13,12°12.12,DM
7,12°12.12',DM
12,12° 12.12,DM
14,"12° 12'12.12""",DMS


Apply the same format check to the actual data

In [13]:
cgg_df['lat_format'] = cgg_df.cleaned_lat.map(check_format, na_action='ignore')

Manual check that the lat format identification was done correctly

In [ ]:
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    print(lat_formats)
    print()

DD
0     dd.dd
1    dd.dd°
2      d.dd
dtype: object

nan
Series([], dtype: object)

invalid format
0          dddddddd
1            dddddd
2    ddd dd' dd.dd"
3            ddd.dd
4             ddddd
5        dd°dd'ddd"
6        dd' dd.dd"
dtype: object

DM
0      dd dd.dd
1        dd°dd'
2     dd°dd.dd'
3     dd° dd.dd
4      dd°dd.dd
5    dd° dd.dd'
6      d°dd.dd'
dtype: object

DMS
0        dd dd dd.dd
1       dd°dd'dd.dd"
2          dd°dd'dd"
3           dd°d'dd"
4        dd°d'dd.dd"
5         dd dd' dd"
6     dd° dd' dd.dd"
7        dd°dd'd.dd"
8      dd° d' dd.dd"
9      dd° dd'dd.dd"
10       d°dd'dd.dd"
11          dd°dd'd"
12         dd° d'dd"
13       dd° dd' dd"
dtype: object



### Parse

Remove trailing non-numbers to make parsing easier

In [16]:
cgg_df.cleaned_lat = cgg_df.cleaned_lat.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')  # removes traling non numbers

##### Split lat data into degree minute and decimal second

If it looks good apply the splitting to the data

In [20]:
cgg_df['cleaned_lat_split'] = cgg_df.cleaned_lat.str.split(pat=r"[ °']+", regex=True)

Manually inspect the splitting

In [21]:
cgg_df['cleaned_lat_split']

0          [61.1399]
1          [61.1399]
2          [61.1399]
3          [61.1399]
4          [61.1399]
            ...     
20852    [64, 33.10]
20853    [64, 33.10]
20854    [64, 33.10]
20855    [64, 33.10]
20856    [64, 33.10]
Name: cleaned_lat_split, Length: 20857, dtype: object

In [18]:
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    
    dms_formats_split = lat_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lat_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

DD
dd.dd
['dd.dd']
d.dd
['d.dd']

nan

invalid format
dddddddd
['dddddddd']
dddddd
['dddddd']
ddd dd' dd.dd
['ddd', 'dd', 'dd.dd']
ddd.dd
['ddd.dd']
ddddd
['ddddd']
dd°dd'ddd
['dd', 'dd', 'ddd']
dd' dd.dd
['dd', 'dd.dd']

DM
dd dd.dd
['dd', 'dd.dd']
dd°dd
['dd', 'dd']
dd°dd.dd
['dd', 'dd.dd']
dd° dd.dd
['dd', 'dd.dd']
d°dd.dd
['d', 'dd.dd']

DMS
dd dd dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'dd
['dd', 'dd', 'dd']
dd°d'dd
['dd', 'd', 'dd']
dd°d'dd.dd
['dd', 'd', 'dd.dd']
dd dd' dd
['dd', 'dd', 'dd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'd.dd
['dd', 'dd', 'd.dd']
dd° d' dd.dd
['dd', 'd', 'dd.dd']
dd° dd'dd.dd
['dd', 'dd', 'dd.dd']
d°dd'dd.dd
['d', 'dd', 'dd.dd']
dd°dd'd
['dd', 'dd', 'd']
dd° d'dd
['dd', 'd', 'dd']
dd° dd' dd
['dd', 'dd', 'dd']



Test if the different formats have correct number of elementes in the split lists and that each element only contains numbers

In [22]:
def all_numbers(lst):
    return all(s.replace(".", "", 1).isdigit() for s in lst)

for i, row in cgg_df[['lat_format', 'cleaned_lat_split']].iterrows():
    split_lst = row.cleaned_lat_split
    lat_format = row.lat_format
    
    if lat_format == 'DD':
        assert len(split_lst) == 1
    if lat_format == 'DM':
        assert len(split_lst) == 2
    if lat_format == 'DMS':
        assert len(split_lst) == 3
    if isinstance(split_lst, list):
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')

Convert Split DM data into decimal degrees

In [43]:
def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
    lat_format = row.lat_format 
    split_lst = row.cleaned_lat_split
    if lat_format == 'DD':
        return convert_dd(split_lst)
    elif lat_format == 'DM':
        return convert_dm(split_lst)
    elif lat_format == 'DMS':
        return convert_dms(split_lst)
    else:
        return np.nan
    
cgg_df['converted_lat'] = cgg_df.apply(convert_to_dd, axis=1)

In [45]:
cgg_df.converted_lat

0        61.1399
1        61.1399
2        61.1399
3        61.1399
4        61.1399
          ...   
20852        NaN
20853        NaN
20854        NaN
20855        NaN
20856        NaN
Name: converted_lat, Length: 20857, dtype: float64

Check that the conversion was done correctly

In [48]:
lat_formats = (cgg_df.converted_lat
 .map(lambda x: re.sub(r'\d+', '12', str(x)), na_action='ignore')  # Turns all n sized numbers into 123
 )
lat_formats.unique()

array(['12.12', nan], dtype=object)

Manually inspect the data to verify

In [52]:
cgg_df[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format']].sample(50)

,Lat,cleaned_lat,converted_lat,lat_format
12087,78.4892109999999,78.4892109999999,78.489211,DD
16785,55.2873,55.2873,55.287300,DD
15436,40.57858,40.57858,40.578580,DD
53,"+61,1399",61.1399,61.139900,DD
8895,64.62147˚N,64.62147,64.621470,DD
11853,NaN,NaN,NaN,NaN
10860,NaN,NaN,NaN,NaN
10420,"34°35'2.19""N",34°35'2.19,34.583942,DMS
9111,79.3382˚N,79.3382,79.338200,DD
6146,NaN,NaN,NaN,NaN


### Clean invalid formatted coordinates

Check if any lat directions are bad

Mark rows that contain E, W, e or w in lat as invalid, bad_direction

In [14]:
invalid_filter = cgg_df.lat_direction.str.contains(r'E|W|e|w', na=False)

In [15]:
cgg_df[invalid_filter]['invalid_format'] = True
cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'

C:\Users\glj523\AppData\Local\Temp\ipykernel_19008\311128983.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format'] = True
C:\Users\glj523\AppData\Local\Temp\ipykernel_19008\311128983.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'


Do format checking on invalid ones, replace digits with d

Lats that are probably missing a decimal

In [16]:
no_dec_filter = cgg_df.cleaned_lat.map(lambda x: bool(re.match(pattern=r'^\d+$', string=x)), na_action='ignore')
sorted(cgg_df[no_dec_filter].cleaned_lat.unique())

ValueError: Cannot mask with non-boolean array containing NA / NaN values